In [ ]:
from pathlib import Path

import numpy as np
from alkaid.codegen import RTLModel
from alkaid.converter import trace_model
from alkaid.trace import FVArray, trace
from fqtree import FQTreeClassifier
from sklearn.datasets import fetch_openml

data = fetch_openml('mnist_784', version=1, as_frame=False)

X, y = data.data, data.target
y = y.astype(np.int64)
(X_train, y_train), (X_test, y_test) = (X[:60000], y[:60000]), (X[60000:], y[60000:])
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
X_train_q = np.floor(X_train / 2**7)
X_test_q = np.floor(X_test / 2**7)

In [2]:
def run(n_estimator, max_depth, scale, bias):
    Path('/tmp/fqtree/').mkdir(exist_ok=True)
    model = FQTreeClassifier(
        scale=scale,
        bias=bias,
        num_class=10,
        n_estimators=n_estimator,
        max_depth=max_depth,
        eta=0.8,
    )
    model.fit(X_train_q, y_train, eval_set=[(X_test_q, y_test)], verbose=False)
    print(f'Software acc: {np.mean(model.predict(X_test_q) == y_test):.4f}')

    hw_bst = model.ibooster()
    inp = FVArray.new(28**2).quantize(0, 1, 0).as_new()
    _, out = trace_model(hw_bst, inputs=inp, mode='mux')
    comb = trace(inp, out)

    train_acc = np.mean(np.argmax(comb.predict(X_train_q), axis=1) == y_train)
    test_acc = np.mean(np.argmax(comb.predict(X_test_q), axis=1) == y_test)
    print(f'HW Train acc: {train_acc:.4f}, Test acc: {test_acc:.4f}')
    # print(f'Estimated LUT: {comb.cost:.1f}')

    rtl = RTLModel(
        comb,
        f'/tmp/fqtree/mnist-{n_estimator=}-{max_depth=}-{scale=}-{bias=}',
        'model',
        n_stages=2,
        clock_period=1.6,
        clock_uncertainty=0,
        part_name='xcvu9p-flgb2104-2-i',
    )
    rtl.write(xls_opt=True, metadata={'comb_metric': test_acc})
    for i in range(4):
        try:
            print('retry...') if i > 0 else None
            rtl._compile(_env={'VERILATOR_FLAGS': ''}, nproc=4)
            break
        except Exception as _e:
            pass  # verilator internal error
    else:
        raise RuntimeError('Failed to compile RTL model after 4 attempts')
    assert np.all(rtl.predict(X_test_q) == comb.predict(X_test_q))  # bit-exact check

In [3]:
run(64, 6, 2.5, -1)

Software acc: 0.9770
HW Train acc: 1.0000, Test acc: 0.9768


In [4]:
run(48, 4, 1.5, -1)

Software acc: 0.9675
HW Train acc: 0.9948, Test acc: 0.9671


In [5]:
run(32, 4, 1.5, -1)

Software acc: 0.9562
HW Train acc: 0.9810, Test acc: 0.9565
